In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, StringType, DoubleType, BooleanType
from pyspark.sql.functions import *

In [3]:
#SESSION
spark = SparkSession.builder \
    .appName("Week1_MiniProject") \
    .master("local[*]") \
    .getOrCreate()

In [4]:
#CONFIG
INPUT_PATH = "employees_raw.csv"
OUTPUT_CLEANED = "output/employees_cleaned"
OUTPUT_SUMMARY = "output/department_summary"

In [5]:
#SCHEMA
schema = StructType([
        StructField("emp_id", IntegerType(), False),
        StructField("name", StringType(), False),
        StructField("department",StringType(), True),
        StructField("salary", DoubleType(), True),
        StructField("location", StringType(), True),
        StructField("hire_date", StringType(), True),
        StructField("is_active", BooleanType(), True),
])

In [6]:
#LOAD
df_raw = spark.read.csv(INPUT_PATH, header=True, schema=schema)
print(f"row count is: {df_raw.count()}")

row count is: 15


In [7]:
#CLEANING
df_active = df_raw.filter(col("is_active")==True)
df_filled = df_active.na.fill({"department": "Unassigned", "location": "Unknown"})
dept_avg = df_filled.groupBy("department").agg(avg("salary").alias("avg_Sal"))
df_clean = (
    df_filled.join(dept_avg, "department", "left")
    .withColumn("salary", coalesce(col("salary"),col("avg_sal")))
    .drop("avg_sal")
    .na.drop(subset=["emp_id","name","salary"])
)

In [8]:
#TRANSFORM
df_final = (
    df_clean
    .withColumn("salary_category",
               when(col("salary")>80000, "High")
               .when(col("salary")>50000, "Medium")
               .otherwise("Low")
               )
    .withColumn("tax_rate",
               when(col("salary")>80000,0.30)
               .when(col("salary")>50000,0.25)
               .otherwise(0.20)
               )
    .withColumn("tax_amount",round(col("salary")* col("tax_rate"),2))
    .withColumn("net_salary",round(col("salary")- col("tax_amount"),2))
    .withColumn("years_at_company" , round(date_diff(current_date(), to_date(col("hire_date")))/365.25,1))
)

In [12]:
#AGGREGATE
summary = (
    df_final
    .groupBy("department")
    .agg(
        count("*").alias("headcount"),
        round(avg(col("salary")), 0).alias("avg_salary"),
        max("salary").alias("max_salary"),
        min("salary").alias("min_salary"),
        round(sum("salary"), 0).alias("total_payroll"),
        count(when(col("salary_category") == "High", True)).alias("high_earners")
    )
    .withColumn(
        "high_earner_pct",
        round(col("high_earners") / col("headcount") * 100, 1)
    )
    .orderBy(col("avg_salary").desc())
)

In [13]:
#OUTPUT
df_final.write.csv(OUTPUT_CLEANED, header=True, mode="overwrite")
summary.write.csv(OUTPUT_SUMMARY, header= True, mode="overwrite")

print(f"Clean Rows: {df_final.count()}")
summary.show(truncate=False)
print("Done😎")

Clean Rows: 14
+-----------+---------+----------+----------+----------+-------------+------------+---------------+
|department |headcount|avg_salary|max_salary|min_salary|total_payroll|high_earners|high_earner_pct|
+-----------+---------+----------+----------+----------+-------------+------------+---------------+
|Engineering|4        |90000.0   |110000.0  |72000.0   |360000.0     |3           |75.0           |
|Data       |5        |81500.0   |95000.0   |71000.0   |407500.0     |3           |60.0           |
|Unassigned |2        |51500.0   |55000.0   |48000.0   |103000.0     |0           |0.0            |
|HR         |3        |48500.0   |52000.0   |45000.0   |145500.0     |0           |0.0            |
+-----------+---------+----------+----------+----------+-------------+------------+---------------+

Done😎
